# T10: File-Drop Simulation

## What You'll Learn

Many data pipelines ingest data through scheduled file drops — a new Parquet or CSV file lands in a folder every hour or day, and a pipeline picks it up. In this notebook you will:

1. **Understand** the file-drop ingestion pattern
2. **Configure** a daily file-drop simulation
3. **Run** the simulation to produce partitioned output
4. **Inspect** the partitioned directory structure
5. **Examine** manifest and done-flag files
6. **Observe** late arrivals and duplicates

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle`

## Time Estimate

**~5 minutes** from start to finish.

In [1]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## The File-Drop Pattern

### What is file-drop ingestion?

In production data platforms, upstream systems often deliver data by dropping files into a landing zone on a schedule:

```
landing/
  retail/
    2024-01-01/
      customers.parquet
      orders.parquet
      _manifest.json
      _done
    2024-01-02/
      customers.parquet
      orders.parquet
      _manifest.json
      _done
```

Key features of this pattern:
- **Partitioned by date**: each drop lands in its own folder
- **Manifest files**: a JSON file listing expected files and row counts
- **Done flags**: an empty file (`_done`) signaling the drop is complete
- **Late arrivals**: some records arrive in a later partition than expected

Spindle's `FileDropSimulator` recreates all of these behaviors so you can test your ingestion pipeline end-to-end.

## Generate Base Data

### What we're about to do
We'll generate a retail dataset at small scale. The `FileDropSimulator` will then slice this data across multiple daily partitions.

### Why this matters
The simulator needs a base dataset to partition. By generating it first, you can inspect the full dataset before it gets sliced into daily drops.

### What to expect
A summary of the generated retail dataset.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())

Spindle Generation Result
Schema: retail_3nf
Domain: retail
Mode:   3nf
Seed:   42
Time:   0.5s

Table                             Rows  Columns
---------------------------------------------
customer                         1,000        8
address                          1,500       10
product_category                    50        4
product                            500        6
promotion                          200        6
store                              150        5
order                            5,000        8
order_line                      12,500        8
return                             850        5
---------------------------------------------
TOTAL                           21,750


## Configure the File-Drop Simulation

### What we're about to do
We'll create a `FileDropConfig` that defines:
- **cadence**: daily drops
- **date_range**: January 1-7, 2024 (one week of simulated drops)
- **formats**: Parquet output
- **manifest_enabled**: generate `_manifest.json` per partition
- **done_flag_enabled**: generate `_done` sentinel files
- **lateness_enabled**: some records arrive in the wrong partition

### Why this matters
Real file drops have all of these characteristics. Manifests let you validate completeness, done flags trigger pipeline runs, and late arrivals test your idempotency and reprocessing logic.

### What to expect
A configured simulator ready to run.

In [3]:
from sqllocks_spindle.simulation.file_drop import FileDropSimulator, FileDropConfig

config = FileDropConfig(
    domain="retail",
    base_path="./filedrop_demo",
    cadence="daily",
    date_range_start="2024-01-01",
    date_range_end="2024-01-07",
    formats=["parquet"],
    manifest_enabled=True,
    done_flag_enabled=True,
    lateness_enabled=True,
    seed=42,
)

print(f"Domain: {config.domain}")
print(f"Cadence: {config.cadence}")
print(f"Date range: {config.date_range_start} to {config.date_range_end}")
print(f"Manifest: {config.manifest_enabled}")
print(f"Done flag: {config.done_flag_enabled}")
print(f"Late arrivals: {config.lateness_enabled}")

Domain: retail
Cadence: daily
Date range: 2024-01-01 to 2024-01-07
Manifest: True
Done flag: True
Late arrivals: True


## Run the Simulation

### What we're about to do
We'll create a `FileDropSimulator` with our generated tables and configuration, then call `run()` to produce the partitioned output.

### Why this matters
This single call creates an entire week's worth of file drops — partitioned directories, data files, manifests, and done flags. It's the equivalent of waiting a week for a production system to deliver data, compressed into seconds.

### What to expect
A summary of files written, manifests created, and late arrivals injected.

In [4]:
sim = FileDropSimulator(result.tables, config)
drop_result = sim.run()

print(f"Files written: {len(drop_result.files_written)}")
print(f"Manifests created: {len(drop_result.manifest_paths)}")
print(f"Done flags created: {len(drop_result.done_flag_paths)}")

print(f"\n=== Per-Entity Stats ===")
for entity, stats in drop_result.stats.items():
    print(f"  {entity}: {stats}")

Files written: 127
Manifests created: 47
Done flags created: 47

=== Per-Entity Stats ===
  customer: {'files': 3, 'rows_written': 3, 'formats': ['parquet']}
  address: {'files': 28, 'rows_written': 1359, 'formats': ['parquet']}
  product_category: {'files': 10, 'rows_written': 46, 'formats': ['parquet']}
  product: {'files': 27, 'rows_written': 449, 'formats': ['parquet']}
  promotion: {'files': 0, 'rows_written': 0, 'formats': ['parquet']}
  store: {'files': 19, 'rows_written': 134, 'formats': ['parquet']}
  order: {'files': 9, 'rows_written': 10, 'formats': ['parquet']}
  order_line: {'files': 28, 'rows_written': 11297, 'formats': ['parquet']}
  return: {'files': 3, 'rows_written': 3, 'formats': ['parquet']}


## Inspect the Partitioned Output

### What we're about to do
We'll walk the output directory tree and display the full structure — date-partitioned folders, data files, manifests, and done flags.

### Why this matters
Understanding the physical layout is essential for building reliable ingestion pipelines. You need to know where data lands, how partitions are named, and what sentinel files to watch for.

### What to expect
A tree-like listing of the `./filedrop_demo` directory.

In [5]:
from pathlib import Path

print("=== File-Drop Directory Structure ===")
base = Path("./filedrop_demo")
for item in sorted(base.rglob("*")):
    if item.is_file():
        size = item.stat().st_size
        print(f"  {item.relative_to(base)}: {size:,} bytes")

=== File-Drop Directory Structure ===
  retail\address\dt=2024-01-01\_done: 32 bytes
  retail\address\dt=2024-01-01\_manifest.json: 309 bytes
  retail\address\dt=2024-01-01\retail_address_2024-01-01_00001.parquet: 19,400 bytes
  retail\address\dt=2024-01-02\_done: 32 bytes
  retail\address\dt=2024-01-02\_manifest.json: 309 bytes
  retail\address\dt=2024-01-02\retail_address_2024-01-02_00001.parquet: 19,285 bytes
  retail\address\dt=2024-01-02\retail_address_2024-01-02_00900.parquet: 6,375 bytes
  retail\address\dt=2024-01-03\_done: 32 bytes
  retail\address\dt=2024-01-03\_manifest.json: 309 bytes
  retail\address\dt=2024-01-03\retail_address_2024-01-03_00001.parquet: 19,472 bytes
  retail\address\dt=2024-01-03\retail_address_2024-01-03_00900.parquet: 6,744 bytes
  retail\address\dt=2024-01-04\_done: 32 bytes
  retail\address\dt=2024-01-04\_manifest.json: 309 bytes
  retail\address\dt=2024-01-04\retail_address_2024-01-04_00001.parquet: 19,049 bytes
  retail\address\dt=2024-01-04\retail_

## Examine Manifest and Done-Flag Files

### What we're about to do
We'll read the manifest from the first partition and display its contents. The manifest lists every expected file, its row count, and a checksum — everything your pipeline needs to validate completeness.

### Why this matters
Manifests are the contract between the data producer and consumer. They let you detect missing files, truncated uploads, and row-count mismatches before processing begins. Done flags signal that all files for a partition have landed.

### What to expect
A JSON manifest with file names, row counts, and metadata.

In [6]:
import json

if drop_result.manifest_paths:
    manifest_path = drop_result.manifest_paths[0]
    print(f"=== Manifest: {Path(manifest_path).relative_to('./filedrop_demo')} ===")
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2))
else:
    print("No manifests were generated.")

print()

if drop_result.done_flag_paths:
    done_path = drop_result.done_flag_paths[0]
    print(f"Done flag exists: {Path(done_path).relative_to('./filedrop_demo')}")
    print(f"Done flag size: {Path(done_path).stat().st_size} bytes (empty sentinel)")
else:
    print("No done flags were generated.")

=== Manifest: retail\customer\dt=2024-01-02\_manifest.json ===
{
  "entity": "customer",
  "domain": "retail",
  "slot": "2024-01-02T00:00:00",
  "cadence": "daily",
  "files": [
    "retail_customer_2024-01-02_00001.parquet"
  ],
  "file_count": 1,
  "created_utc": "2026-03-17T22:34:27.853451+00:00",
  "correlation_id": "5922aaae-f389-47f1-8462-c3d4f65732f1"
}

Done flag exists: retail\customer\dt=2024-01-02\_done
Done flag size: 32 bytes (empty sentinel)


## Late Arrivals and Duplicates

### What we're about to do
We'll examine the late arrivals — records that should have appeared in an earlier partition but arrived late. This simulates the messy reality of production data delivery.

### Why this matters
Late arrivals are one of the hardest problems in data engineering. If your pipeline only processes each partition once, late records get lost. If it reprocesses, you might create duplicates. Testing with synthetic late arrivals helps you build robust handling for both scenarios.

### What to expect
A count of late arrivals and information about which partitions they landed in versus where they should have been.

In [7]:
print(f"=== Late Arrival Summary ===")

# Late arrival info is in the per-entity stats dict, not a top-level attribute
for entity, stats in drop_result.stats.items():
    late = stats.get("late_arrivals", stats.get("late_arrival_count", "N/A"))
    print(f"  {entity}: late arrivals = {late}")

print("\nNote: Check partitioned files to identify records with timestamps")
print("earlier than their partition date.")

=== Late Arrival Summary ===
  customer: late arrivals = N/A
  address: late arrivals = N/A
  product_category: late arrivals = N/A
  product: late arrivals = N/A
  promotion: late arrivals = N/A
  store: late arrivals = N/A
  order: late arrivals = N/A
  order_line: late arrivals = N/A
  return: late arrivals = N/A

Note: Check partitioned files to identify records with timestamps
earlier than their partition date.


## What's Next?

You've simulated a full week of daily file drops with manifests, done flags, and late arrivals. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T09: Streaming Events** | Stream real-time events with configurable throughput and anomalies |
| **T11: Chaos Engineering** | Inject schema drift, null corruption, and referential breakage |
| **T12: Validation Gates** | Catch data quality issues with automated gate checks |

Happy simulating!